# Heart Signal Reconstruction & Joint TPE Hyperparameter Optimization Pipeline

This notebook documents, visualizes, and evaluates the end-to-end signal processing and machine learning pipeline to reconstruct the cardiac signal from a **5-dimensional wrist-worn sensor recording** (`[ppg1, ppg2, accx, accy, accz]`) and estimate the Heart Rate in **Beats Per Minute (BPM)** on each individual 8-second window ($N = 1000$ samples at $f_s = 125\text{ Hz}$).

---

## 1. Signal Processing Pipeline

Each 5-dimensional window $\mathbf{X}_w \in \mathbb{R}^{5 \times N}$ is processed **individually** (matching the test-set evaluation protocol without merging adjacent windows during filtering or prediction):

```mermaid
flowchart LR
    A["5D Window Input<br/>[ppg1, ppg2, accx, accy, accz]"] --> B["Stage 1: Preprocessing<br/>Bandpass (0.5-4.0 Hz)<br/>+ Unit-Variance Norm"]
    B --> C1["Primary d1[n] (PPG1)"]
    B --> C2["Primary d2[n] (PPG2)"]
    B --> D["Stage 2: MISO Regressor U<br/>3-Axis Acc Tapped Delays a[n]"]
    C1 & D --> E1["Stage 3a: MISO LMS Branch 1<br/>e1[n] = d1[n] - w1^T a[n]"]
    C2 & D --> E2["Stage 3b: MISO LMS Branch 2<br/>e2[n] = d2[n] - w2^T a[n]"]
    E1 & E2 --> F["Stage 4: Late Fusion<br/>SNR-Weighted Averaging<br/>e_fused[n] = w1*e1 + w2*e2"]
    F --> G["Stage 5: HR Estimation<br/>Zero-Padded Periodogram<br/>or Elgendi Systolic Peaks"]
```

### Stage 1: Preprocessing (`preprocessing.py`)
1. **Bandpass Filtering ($30\text{–}240\text{ BPM} \equiv 0.5\text{–}4.0\text{ Hz}$)**:
   A causal 4th-order Butterworth bandpass filter isolates the physiological cardiac band $[0.5\text{ Hz}, 4.0\text{ Hz}]$, removing baseline wander ($< 0.5\text{ Hz}$) and high-frequency noise ($> 4.0\text{ Hz}$).
2. **Amplitude Normalization**:
   Each channel is normalized to zero mean and unit standard deviation along the time axis:
   $$\tilde{x}_c[n] = \frac{x_c[n] - \mu_c}{\sigma_c + \epsilon}$$

### Stage 2: MISO Reference Signal Construction (`reference_construction.py`)
The 3-axis accelerometer channels $\mathbf{a}_1[n], \mathbf{a}_2[n], \mathbf{a}_3[n]$ ($x, y, z$) are expanded into tapped delay-line vectors of order $M$:
$$\mathbf{a}_k[n] = \begin{bmatrix} a_k[n] & a_k[n-1] & \dots & a_k[n-M+1] \end{bmatrix}^T \in \mathbb{R}^M$$
Stacking the three axes forms the $3M$-dimensional MISO regressor vector $\mathbf{a}[n] = [\mathbf{a}_1^T[n], \mathbf{a}_2^T[n], \mathbf{a}_3^T[n]]^T$, so that the estimated motion artifact is:
$$\hat{m}[n] = \sum_{k=1}^{3} \mathbf{w}_k^T[n]\mathbf{a}_k[n] = \mathbf{w}^T[n]\mathbf{a}[n]$$

### Stage 3: Adaptive MISO LMS Filtering (`LMS_algorithms.py`)
Taking a PPG channel as primary input $d[n] = s[n] + m[n]$ and the triaxial accelerometer as reference $\mathbf{a}[n]$, the cleared cardiac signal is the adaptive filter error output:
$$e[n] = d[n] - \hat{m}[n] = s[n] + m[n] - \mathbf{w}^T[n]\mathbf{a}[n]$$

Three adaptive LMS variants are compared:
- **Standard LMS (`standard_lms`)** — Hyperparameters: $M, \mu$
  $$\mathbf{w}[n+1] = \mathbf{w}[n] + \mu \, e[n] \, \mathbf{a}[n]$$
- **Normalized LMS (`nlms`)** — Hyperparameters: $M, \mu, \delta$
  $$\mathbf{w}[n+1] = \mathbf{w}[n] + \frac{\mu}{\delta + \|\mathbf{a}[n]\|^2} \, e[n] \, \mathbf{a}[n]$$
- **Variable Step-Size LMS (`vs_lms`, Kwong & Johnston)** — Hyperparameters: $M, \mu_{\max}, \mu_{\min}, \alpha, \gamma$
  $$\mathbf{w}[n+1] = \mathbf{w}[n] + \mu[n] \, e[n] \, \mathbf{a}[n], \qquad \mu[n+1] = \text{clip}\!\left(\alpha \mu[n] + \gamma e^2[n],\, \mu_{\min},\, \mu_{\max}\right)$$

### Stage 4: Dual PPG Late Fusion via SNR-Weighted Averaging (`fusion.py`)
Both PPG channels (`ppg1` and `ppg2`) are filtered **in parallel** to produce $e_1[n]$ and $e_2[n]$. Their in-band cardiac spectral Signal-to-Noise Ratios ($\text{SNR}_1, \text{SNR}_2$) are computed around the fundamental cardiac peak (and 2nd harmonic), and the two branches are fused via SNR-Weighted Averaging:
$$e_{\text{fused}}[n] = \frac{\text{SNR}_1}{\text{SNR}_1 + \text{SNR}_2} e_1[n] + \frac{\text{SNR}_2}{\text{SNR}_1 + \text{SNR}_2} e_2[n]$$

### Stage 5: Window-Level Heart Rate Estimation (`bpm_detection.py`)
1. **Zero-Padded Periodogram (`zero_padded_periodogram_bpm`)** — Hyperparameters: `nfft`, `window_type`
   Applies a spectral taper (`hann`, `hamming`, or `blackman`), explicitly zero-pads the $N$-sample window to `nfft` samples, computes the periodogram, and converts the peak frequency in $[0.5, 4.0]\text{ Hz}$ to BPM ($\widehat{\text{BPM}} = 60 \cdot f_{\text{peak}}$).
2. **Systolic Peak Detection — Elgendi's Algorithm (`elgendi_systolic_peak_bpm`)** — Hyperparameters: `w1_sec`, `w2_sec`, `beta_offset`
   Squares the positive part of the signal and compares a short systolic moving average ($\text{MA}_{\text{peak}}$, window $W_1$) against a heartbeat moving average ($\text{MA}_{\text{beat}}$, window $W_2$) plus dynamic offset $\beta \cdot \overline{S}$ to isolate systolic peaks and compute BPM from the median inter-beat interval.

---

## 2. Outlier-Penalized Composite Objective & 2-Phase TPE Optimization (`optimization.py`)

To heavily penalize catastrophic tracking errors (e.g. locking onto motion harmonics $> 10\text{ BPM}$ off ground-truth), the objective function combines **Average Absolute Error (AAE)**, **RMSE**, and a **soft hinge penalty on outliers** ($|\hat{y}_w - y_w| > \tau_{\text{out}}$, with $\tau_{\text{out}} = 10.0\text{ BPM}$):

$$\mathcal{L}(\hat{\mathbf{y}}, \mathbf{y}) = \underbrace{\frac{1}{W}\sum_{w=1}^{W} |\hat{y}_w - y_w|}_{\text{AAE}} + 0.35 \underbrace{\sqrt{\frac{1}{W}\sum_{w=1}^{W} (\hat{y}_w - y_w)^2}}_{\text{RMSE}} + 0.50 \underbrace{\frac{1}{W}\sum_{w=1}^{W} \max\!\left(0,\, |\hat{y}_w - y_w| - 10.0\right)}_{\text{Outlier Hinge Penalty}}$$

- **Validation Protocol**: Subject-Wise Group K-Fold (`subject_wise_group_kfold`) partitions windows strictly by subject ID.
- **Phase 1 (Decoupled Fast Screening)**: Screens LMS filter configurations (`standard_lms`, `nlms`, `vs_lms`) using the fast Spectral SNR Proxy.
- **Phase 2 (End-to-End TPE with Pruning)**: Warm-starts all 6 `(LMS x HR)` combinations and jointly optimizes the conditional tree across Subject-Wise Group K-Fold splits using `MedianFoldPruner`.

In [ ]:
import sys
from pathlib import Path
from pprint import pprint
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is on sys.path whether notebook is run from root or restructured_implementation/
current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name == "restructured_implementation" else current_dir
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from restructured_implementation import (
    FS,
    WIN,
    STEP,
    load_ts,
    get_participant_groups,
    bandpass,
    normalize_amplitude,
    lms_bucket,
    optimize_hr_pipeline,
    objective_function,
)

## 3. Load Training and Test Datasets

We load `IEEEPPG_TRAIN.ts` and `IEEEPPG_TEST.ts` as individual 8-second windows of shape `(5, 1000)` and identify the recording subject IDs.

In [ ]:
train_path = project_root / "IEEEPPG_TRAIN.ts"
test_path = project_root / "IEEEPPG_TEST.ts"

X_train, y_train = load_ts(str(train_path))
X_test, y_test = load_ts(str(test_path))

train_groups = get_participant_groups(X_train)
test_groups = get_participant_groups(X_test)

print(f"Training set : X_train = {X_train.shape}, y_train = {y_train.shape} | Subjects = {len(np.unique(train_groups))}")
print(f"Test set     : X_test  = {X_test.shape}, y_test  = {y_test.shape} | Subjects = {len(np.unique(test_groups))}")

## 4. Visual Walkthrough: How a Single Window Changes After Each Pipeline Step

Below we select a single 8-second window (`sample_idx = 0`) and visualize how the 5-dimensional signal transforms step-by-step through:
1. **Raw Sensor Signals** (`ppg1, ppg2` and 3-axis accelerometer `accx, accy, accz`)
2. **Bandpass Filtering** ($0.5\text{–}4.0\text{ Hz} \equiv 30\text{–}240\text{ BPM}$)
3. **Amplitude Normalization** (zero-mean, unit-variance primary inputs $d_1[n], d_2[n]$ and reference $\mathbf{a}[n]$)
4. **Parallel Adaptive MISO LMS Filtering** (motion artifact estimates $\hat{m}_1[n], \hat{m}_2[n]$ subtracted from $d_1[n], d_2[n]$ to yield $e_1[n], e_2[n]$)
5. **SNR-Weighted Late Fusion & Spectral HR Detection** ($e_{\text{fused}}[n] = w_1 e_1[n] + w_2 e_2[n]$)

In [ ]:
sample_idx = 0
raw_win = X_train[sample_idx]          # Shape: (5, 1000)
true_bpm_sample = y_train[sample_idx]
t_axis = np.arange(WIN) / FS           # Time axis in seconds (0 to 8 s)

# Step 1a: Bandpass filter (0.5 - 4.0 Hz)
bp_win = bandpass(raw_win, lo=0.5, hi=4.0, fs=FS)

# Step 1b: Amplitude normalization
norm_win = normalize_amplitude(bp_win)
d1, d2, acc_norm = norm_win[0], norm_win[1], norm_win[2:5]

# Step 2, 3 & 4: Parallel MISO LMS Filtering + SNR-Weighted Late Fusion
e_fused_sample, sample_details = lms_bucket(
    d1, d2, acc_norm, algorithm="vs_lms", M=16, return_details=True
)
e1_sample = sample_details["e1"]
e2_sample = sample_details["e2"]
m_hat1_sample = sample_details["m_hat1"]
m_hat2_sample = sample_details["m_hat2"]
f_info = sample_details["fusion_info"]

fig, axes = plt.subplots(5, 2, figsize=(15, 14), sharex="col")

# Row 0: Raw Signals
axes[0, 0].plot(t_axis, raw_win[0], label="Raw PPG1", color="tab:red", alpha=0.85)
axes[0, 0].plot(t_axis, raw_win[1], label="Raw PPG2", color="tab:orange", alpha=0.85)
axes[0, 0].set_title("Step 0: Raw Dual PPG Channels")
axes[0, 0].legend(loc="upper right", fontsize=8); axes[0, 0].grid(alpha=0.3)

for k, (lbl, col) in enumerate(zip(["Acc X", "Acc Y", "Acc Z"], ["tab:blue", "tab:green", "tab:purple"])):
    axes[0, 1].plot(t_axis, raw_win[2 + k], label=lbl, color=col, alpha=0.8)
axes[0, 1].set_title("Step 0: Raw 3-Axis Accelerometer")
axes[0, 1].legend(loc="upper right", fontsize=8); axes[0, 1].grid(alpha=0.3)

# Row 1: Bandpass Filtered (0.5 - 4.0 Hz / 30 - 240 BPM)
axes[1, 0].plot(t_axis, bp_win[0], label="Bandpass PPG1", color="tab:red", alpha=0.85)
axes[1, 0].plot(t_axis, bp_win[1], label="Bandpass PPG2", color="tab:orange", alpha=0.85)
axes[1, 0].set_title("Step 1a: After Bandpass Filter (0.5 - 4.0 Hz / 30 - 240 BPM)")
axes[1, 0].legend(loc="upper right", fontsize=8); axes[1, 0].grid(alpha=0.3)

for k, (lbl, col) in enumerate(zip(["BP Acc X", "BP Acc Y", "BP Acc Z"], ["tab:blue", "tab:green", "tab:purple"])):
    axes[1, 1].plot(t_axis, bp_win[2 + k], label=lbl, color=col, alpha=0.8)
axes[1, 1].set_title("Step 1a: Bandpass-Filtered Accelerometer")
axes[1, 1].legend(loc="upper right", fontsize=8); axes[1, 1].grid(alpha=0.3)

# Row 2: Amplitude Normalized
axes[2, 0].plot(t_axis, d1, label="Normalized d1[n] (PPG1)", color="tab:red", alpha=0.85)
axes[2, 0].plot(t_axis, d2, label="Normalized d2[n] (PPG2)", color="tab:orange", alpha=0.85)
axes[2, 0].set_title("Step 1b: After Amplitude Normalization (Unit Variance)")
axes[2, 0].legend(loc="upper right", fontsize=8); axes[2, 0].grid(alpha=0.3)

for k, (lbl, col) in enumerate(zip(["Norm a_x[n]", "Norm a_y[n]", "Norm a_z[n]"], ["tab:blue", "tab:green", "tab:purple"])):
    axes[2, 1].plot(t_axis, acc_norm[k], label=lbl, color=col, alpha=0.8)
axes[2, 1].set_title("Step 1b: Normalized Reference Input a[n]")
axes[2, 1].legend(loc="upper right", fontsize=8); axes[2, 1].grid(alpha=0.3)

# Row 3: Parallel Adaptive MISO LMS Filtering (Branch 1 & Branch 2)
axes[3, 0].plot(t_axis, d1, label="Primary d1[n]", color="lightgray", lw=1.2)
axes[3, 0].plot(t_axis, m_hat1_sample, label="Motion Estimate m_hat1[n]", color="tab:purple", ls="--", lw=1.1)
axes[3, 0].plot(t_axis, e1_sample, label="Cleaned e1[n] = d1 - m_hat1", color="tab:red", lw=1.3)
axes[3, 0].set_title(f"Step 3 (Branch 1): Parallel MISO LMS on PPG1 (SNR1 = {f_info['snr1']:.2f})")
axes[3, 0].legend(loc="upper right", fontsize=8); axes[3, 0].grid(alpha=0.3)

axes[3, 1].plot(t_axis, d2, label="Primary d2[n]", color="lightgray", lw=1.2)
axes[3, 1].plot(t_axis, m_hat2_sample, label="Motion Estimate m_hat2[n]", color="tab:purple", ls="--", lw=1.1)
axes[3, 1].plot(t_axis, e2_sample, label="Cleaned e2[n] = d2 - m_hat2", color="tab:orange", lw=1.3)
axes[3, 1].set_title(f"Step 3 (Branch 2): Parallel MISO LMS on PPG2 (SNR2 = {f_info['snr2']:.2f})")
axes[3, 1].legend(loc="upper right", fontsize=8); axes[3, 1].grid(alpha=0.3)

# Row 4: SNR-Weighted Late Fusion Output
axes[4, 0].plot(t_axis, e1_sample, label=f"e1[n] (w1={f_info['w1']:.2f})", color="tab:red", alpha=0.45)
axes[4, 0].plot(t_axis, e2_sample, label=f"e2[n] (w2={f_info['w2']:.2f})", color="tab:orange", alpha=0.45)
axes[4, 0].plot(t_axis, e_fused_sample, label="Fused Cardiac Signal e_fused[n]", color="black", lw=1.6)
axes[4, 0].set_title(f"Step 4: Late Fusion Output (True HR = {true_bpm_sample:.1f} BPM)")
axes[4, 0].set_xlabel("Time [s]")
axes[4, 0].legend(loc="upper right", fontsize=8); axes[4, 0].grid(alpha=0.3)

# Compare Pre-LMS vs Post-Fusion on the right bottom panel
axes[4, 1].plot(t_axis, 0.5 * (d1 + d2), label="Before LMS (Unfiltered PPG Mean)", color="tab:gray", ls="--", alpha=0.7)
axes[4, 1].plot(t_axis, e_fused_sample, label="After MISO LMS + Late Fusion", color="tab:blue", lw=1.5)
axes[4, 1].set_title("Comparison: Before vs. After Adaptive LMS & Late Fusion")
axes[4, 1].set_xlabel("Time [s]")
axes[4, 1].legend(loc="upper right", fontsize=8); axes[4, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Joint 2-Phase TPE Optimization & Comparison of All Algorithms

We execute `optimize_hr_pipeline` on `(X_train, y_train)` and compare the performance of:
- **LMS Variants**: `standard_lms`, `nlms`, and `vs_lms` (in Phase 1 Spectral SNR screening and Phase 2 Cross-Validation).
- **Heart Rate Detection Variants**: `periodogram` (Zero-Padded Periodogram) and `elgendi` (Systolic Peak Detection).

In [ ]:
optimization_results = optimize_hr_pipeline(
    X=X_train,
    y=y_train,
    n_splits=5,
    phase1_trials=25,
    phase1_sample_size=120,
    phase1_top_k=6,
    phase2_trials=30,
    prune_percentile=50.0,
    seed=42,
)

best_model = optimization_results["best_model"]
best_hyperparameters = optimization_results["best_hyperparameters"]
best_cv_score = optimization_results["best_score"]
comparison = optimization_results["comparison_summary"]

print("=== Winning Conditional Tree Hyperparameters ===")
pprint(best_hyperparameters)
print(f"\nBest Cross-Validated Composite Objective Score (Train): {best_cv_score:.3f}")

In [ ]:
# Display table and bar charts comparing all evaluated LMS and HR algorithms
phase1_by_lms = comparison["phase1_by_lms"]
phase2_by_pair = comparison["phase2_by_pair"]

print("=== Phase 1: Best Spectral SNR Proxy by LMS Variant ===")
for lms_name, entry in sorted(phase1_by_lms.items()):
    print(f"  {lms_name:14s} | Mean Spectral SNR = {entry['mean_snr']:.4f} | Params = {entry['lms_params']}")

print("\n=== Phase 2: Best Cross-Validation Performance by (LMS + HR Detector) Pair ===")
print(f"{'Algorithm Combination':30s} | {'Objective Score':>15s} | {'CV AAE [BPM]':>13s} | {'Folds':>5s}")
print("-" * 72)
for pair_name, entry in sorted(phase2_by_pair.items(), key=lambda kv: kv[1]["score_objective"]):
    print(f"{pair_name:30s} | {entry['score_objective']:15.3f} | {entry['score_aae']:13.3f} | {entry['folds_evaluated']:5d}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

# Left plot: Phase 1 Spectral SNR by LMS algorithm
lms_names = list(phase1_by_lms.keys())
lms_snrs = [phase1_by_lms[k]["mean_snr"] for k in lms_names]
axes[0].bar(lms_names, lms_snrs, color=["tab:blue", "tab:orange", "tab:green"][:len(lms_names)], edgecolor="black", alpha=0.85)
axes[0].set_title("Phase 1 Screening: Best Spectral SNR Proxy by LMS Variant")
axes[0].set_ylabel("Mean In-Band Cardiac SNR (Higher is Better)")
axes[0].grid(axis="y", alpha=0.3)

# Right plot: Phase 2 Cross-Validated Objective Score & AAE across Algorithm Pairs
pair_labels = list(phase2_by_pair.keys())
obj_vals = [phase2_by_pair[k]["score_objective"] for k in pair_labels]
aae_vals = [phase2_by_pair[k]["score_aae"] for k in pair_labels]
x_pos = np.arange(len(pair_labels))
width = 0.38

axes[1].bar(x_pos - width / 2, obj_vals, width, label="Composite Objective Score", color="tab:red", edgecolor="black", alpha=0.85)
axes[1].bar(x_pos + width / 2, aae_vals, width, label="Mean Absolute Error (AAE [BPM])", color="tab:blue", edgecolor="black", alpha=0.85)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(pair_labels, rotation=25, ha="right")
axes[1].set_title("Phase 2 Cross-Validation: Compared Algorithm Combinations")
axes[1].set_ylabel("Error Score (Lower is Better)")
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Test Set Evaluation (`IEEEPPG_TEST.ts`) & Recombined Signal Before vs. After

We now apply `best_model.predict_with_signals(X_test)` once across all individual windows in the test set. Using these outputs, we first visualize a **recombined multi-window signal segment before and after** adaptive filtering & fusion, and then evaluate the test scores by person.

In [ ]:
# Single-pass prediction on X_test returning both BPM estimates and cleaned window signals
y_test_pred, e_test_cleaned = best_model.predict_with_signals(X_test)

# Overall Test Set Metrics
test_abs_err = np.abs(y_test_pred - y_test)
test_obj_score = objective_function(y_test_pred, y_test)
test_aae = float(np.mean(test_abs_err))
test_rmse = float(np.sqrt(np.mean(test_abs_err ** 2)))
outlier_mask = test_abs_err > 10.0
outlier_fraction = float(np.mean(outlier_mask))

print("=== Final Held-Out Test Set Summary ===")
print(f"Optimal LMS Algorithm    : {best_hyperparameters['lms_algorithm']}")
print(f"Optimal HR Detector      : {best_hyperparameters['hr_algorithm']}")
print(f"Composite Objective Score: {test_obj_score:.3f}")
print(f"Test Set AAE             : {test_aae:.3f} BPM")
print(f"Test Set RMSE            : {test_rmse:.3f} BPM")
print(f"Outlier Fraction (>10BPM): {outlier_fraction:.4f} ({100.0 * outlier_fraction:.2f}% | {int(np.sum(outlier_mask))}/{len(y_test)} windows)")

### Diagram of the Recombined Signals Before and After Pipeline Processing

Consecutive 8-second windows from the same subject advance by `STEP = 250` samples (2 seconds). By concatenating the first window with the new 250-sample stride of subsequent windows from Subject 0, we can inspect the continuous recombined waveform **before** adaptive LMS filtering/fusion and **after** pipeline reconstruction.

In [ ]:
# Recombine consecutive windows from Subject 0 (first 6 consecutive windows = 18 seconds)
subj0_indices = np.where(test_groups == 0)[0][:6]

# Before pipeline: normalized bandpass PPG mean per window
before_windows = [
    0.5 * (normalize_amplitude(bandpass(X_test[idx, :2]))[0] + normalize_amplitude(bandpass(X_test[idx, :2]))[1])
    for idx in subj0_indices
]
# After pipeline: cleaned & SNR-fused cardiac signal per window
after_windows = [e_test_cleaned[idx] for idx in subj0_indices]

recombined_before = np.concatenate([before_windows[0]] + [w[-STEP:] for w in before_windows[1:]])
recombined_after = np.concatenate([after_windows[0]] + [w[-STEP:] for w in after_windows[1:]])
t_recomb = np.arange(len(recombined_before)) / FS

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

axes[0].plot(t_recomb, recombined_before, color="tab:gray", lw=1.1, label="Recombined Signal BEFORE MISO LMS + Late Fusion")
for b in range(1, len(subj0_indices)):
    axes[0].axvline((WIN + (b - 1) * STEP) / FS, color="tab:red", ls=":", alpha=0.4)
axes[0].set_title("Recombined Multi-Window Signal BEFORE Adaptive Filtering & Late Fusion")
axes[0].set_ylabel("Normalized Amplitude")
axes[0].legend(loc="upper right"); axes[0].grid(alpha=0.3)

axes[1].plot(t_recomb, recombined_after, color="tab:blue", lw=1.2, label="Recombined Signal AFTER MISO LMS + SNR Late Fusion")
for b in range(1, len(subj0_indices)):
    axes[1].axvline((WIN + (b - 1) * STEP) / FS, color="tab:red", ls=":", alpha=0.4, label="Window Stitch Boundary" if b == 1 else None)
axes[1].set_title("Recombined Multi-Window Cardiac Signal AFTER Adaptive MISO LMS & Late Fusion")
axes[1].set_xlabel("Time [s]")
axes[1].set_ylabel("Cleaned Cardiac Signal")
axes[1].legend(loc="upper right"); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Score by Person & Subject-Colored Scatterplot (Estimated vs. Ground-Truth BPM)

Here we break down the test performance **by person (subject ID)** and plot the **Estimated vs. Ground-Truth BPM** scatterplot where all windows belonging to the same subject share the same color.

In [ ]:
unique_test_subjects = np.unique(test_groups)
subject_metrics = []

print(f"{'Subject ID':>10s} | {'Windows':>8s} | {'Objective Score':>15s} | {'AAE [BPM]':>10s} | {'RMSE [BPM]':>10s} | {'Outliers (>10 BPM)':>18s}")
print("-" * 84)

for sid in unique_test_subjects:
    mask = test_groups == sid
    preds_s = y_test_pred[mask]
    true_s = y_test[mask]
    err_s = np.abs(preds_s - true_s)

    obj_s = objective_function(preds_s, true_s)
    aae_s = float(np.mean(err_s))
    rmse_s = float(np.sqrt(np.mean(err_s ** 2)))
    out_frac_s = float(np.mean(err_s > 10.0))

    subject_metrics.append({
        "subject_id": int(sid),
        "n_windows": int(np.sum(mask)),
        "objective": obj_s,
        "aae": aae_s,
        "rmse": rmse_s,
        "outlier_frac": out_frac_s,
    })
    print(f"Subject {int(sid):2d} | {int(np.sum(mask)):8d} | {obj_s:15.3f} | {aae_s:10.3f} | {rmse_s:10.3f} | {100.0 * out_frac_s:16.2f} %")

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
cmap = plt.get_cmap("tab10")

# Left: Scatterplot of Estimated vs. Ground-Truth BPM colored by Subject
for idx, sid in enumerate(unique_test_subjects):
    mask = test_groups == sid
    axes[0].scatter(
        y_test[mask],
        y_test_pred[mask],
        color=cmap(idx % 10),
        alpha=0.65,
        s=22,
        label=f"Subject {int(sid)}",
    )

lims = [min(y_test.min(), y_test_pred.min()) - 5, max(y_test.max(), y_test_pred.max()) + 5]
axes[0].plot(lims, lims, "--", color="black", lw=1.4, label="Ideal (y = x)")
axes[0].plot(lims, [lims[0] + 10, lims[1] + 10], ":", color="tab:red", lw=1.0, label="+/- 10 BPM Outlier Bound")
axes[0].plot(lims, [lims[0] - 10, lims[1] - 10], ":", color="tab:red", lw=1.0)
axes[0].set_xlim(lims)
axes[0].set_ylim(lims)
axes[0].set_title("Estimated vs. Ground-Truth BPM Colored by Subject (Test Set)")
axes[0].set_xlabel("Ground-Truth BPM")
axes[0].set_ylabel("Estimated BPM")
axes[0].legend(loc="upper left", fontsize=8, ncol=2)
axes[0].grid(alpha=0.3)

# Right: Score by Person (Composite Objective Score & AAE per Subject)
subj_labels = [f"Subj {m['subject_id']}" for m in subject_metrics]
subj_obj = [m["objective"] for m in subject_metrics]
subj_aae = [m["aae"] for m in subject_metrics]
x_s = np.arange(len(subj_labels))
bar_w = 0.38

axes[1].bar(x_s - bar_w / 2, subj_obj, bar_w, label="Objective Score", color="tab:red", edgecolor="black", alpha=0.85)
axes[1].bar(x_s + bar_w / 2, subj_aae, bar_w, label="AAE [BPM]", color="tab:blue", edgecolor="black", alpha=0.85)
axes[1].set_xticks(x_s)
axes[1].set_xticklabels(subj_labels, rotation=25, ha="right")
axes[1].set_title("Test Set Error Score by Person (Subject-Wise Breakdown)")
axes[1].set_ylabel("Error Score [BPM]")
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Cleaned Signal Inspection: Well-Detected Window vs. Outlier Window

Finally, we compare the cleaned cardiac waveform $e_{\text{fused}}[n]$ and its Zero-Padded Periodogram spectrum for:
- A **well-detected window** ($|\hat{y}_w - y_w| \approx 0\text{ BPM}$), where the adaptive MISO LMS filter cleanly isolates the cardiac fundamental frequency.
- An **outlier window** ($|\hat{y}_w - y_w| > 10\text{ BPM}$), illustrating how residual motion artifacts or harmonic peaks cause a large estimation error.

In [ ]:
good_idx = int(np.argmin(test_abs_err))
outlier_indices = np.where(outlier_mask)[0]
outlier_idx = int(outlier_indices[np.argmax(test_abs_err[outlier_indices])]) if len(outlier_indices) > 0 else int(np.argmax(test_abs_err))

print(f"Total Test Windows     : {len(y_test)}")
print(f"Outlier Windows (>10)  : {int(np.sum(outlier_mask))} ({100.0 * outlier_fraction:.2f}%)")
print(f"Good Window #{good_idx:4d}      : True = {y_test[good_idx]:.2f} BPM | Est = {y_test_pred[good_idx]:.2f} BPM | Error = {test_abs_err[good_idx]:.2f} BPM")
print(f"Outlier Window #{outlier_idx:4d}   : True = {y_test[outlier_idx]:.2f} BPM | Est = {y_test_pred[outlier_idx]:.2f} BPM | Error = {test_abs_err[outlier_idx]:.2f} BPM")

def compute_spectrum_bpm(sig, fs=FS, nfft=8192):
    centered = (sig - np.mean(sig)) * np.hanning(len(sig))
    psd = np.abs(np.fft.rfft(centered, n=nfft)) ** 2
    freqs_bpm = 60.0 * np.fft.rfftfreq(nfft, d=1.0 / fs)
    mask = (freqs_bpm >= 30.0) & (freqs_bpm <= 240.0)
    return freqs_bpm[mask], psd[mask] / (np.max(psd[mask]) + 1e-12)

f_good, psd_good = compute_spectrum_bpm(e_test_cleaned[good_idx])
f_out, psd_out = compute_spectrum_bpm(e_test_cleaned[outlier_idx])

fig, axes = plt.subplots(2, 2, figsize=(15, 8))

# Top-Left: Good Window Cleaned Time-Domain Signal
axes[0, 0].plot(t_axis, e_test_cleaned[good_idx], color="tab:green", lw=1.3)
axes[0, 0].set_title(f"Good Detected Window #{good_idx} — Cleaned Signal (Err = {test_abs_err[good_idx]:.2f} BPM)")
axes[0, 0].set_xlabel("Time [s]"); axes[0, 0].set_ylabel("Amplitude")
axes[0, 0].grid(alpha=0.3)

# Top-Right: Outlier Window Cleaned Time-Domain Signal
axes[0, 1].plot(t_axis, e_test_cleaned[outlier_idx], color="tab:red", lw=1.3)
axes[0, 1].set_title(f"Outlier Window #{outlier_idx} — Cleaned Signal (Err = {test_abs_err[outlier_idx]:.2f} BPM)")
axes[0, 1].set_xlabel("Time [s]"); axes[0, 1].set_ylabel("Amplitude")
axes[0, 1].grid(alpha=0.3)

# Bottom-Left: Good Window Spectrum
axes[1, 0].plot(f_good, psd_good, color="tab:green", lw=1.4, label="Normalized Periodogram")
axes[1, 0].axvline(y_test[good_idx], color="black", ls="--", lw=1.5, label=f"True BPM ({y_test[good_idx]:.1f})")
axes[1, 0].axvline(y_test_pred[good_idx], color="tab:blue", ls=":", lw=1.8, label=f"Estimated BPM ({y_test_pred[good_idx]:.1f})")
axes[1, 0].set_title(f"Good Window #{good_idx} — Cardiac Band Spectrum")
axes[1, 0].set_xlabel("Frequency [BPM]"); axes[1, 0].set_ylabel("Normalized PSD")
axes[1, 0].legend(); axes[1, 0].grid(alpha=0.3)

# Bottom-Right: Outlier Window Spectrum
axes[1, 1].plot(f_out, psd_out, color="tab:red", lw=1.4, label="Normalized Periodogram")
axes[1, 1].axvline(y_test[outlier_idx], color="black", ls="--", lw=1.5, label=f"True BPM ({y_test[outlier_idx]:.1f})")
axes[1, 1].axvline(y_test_pred[outlier_idx], color="tab:blue", ls=":", lw=1.8, label=f"Estimated BPM ({y_test_pred[outlier_idx]:.1f})")
axes[1, 1].set_title(f"Outlier Window #{outlier_idx} — Cardiac Band Spectrum")
axes[1, 1].set_xlabel("Frequency [BPM]"); axes[1, 1].set_ylabel("Normalized PSD")
axes[1, 1].legend(); axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()